# Lab Task 02 — Effect of Image Filtering on Skin-Lesion Classification
**Student ID: 054**

**Objective:** Investigate how different spatial-domain image-processing filters affect the performance of pretrained deep-learning models on HAM10000 skin-lesion classification.

This notebook is organized into the sections required by the lab handout:
1. Dataset Preparation
2. Model Loading
3. Image Filtering
4. Baseline Experiment
5. Training (baseline + filtered experiments)
6. Evaluation
7. Visualization
8. Comparative Analysis
9. Answers to Lab Questions

> **Before running:** fill in the three "best" pretrained models identified in Lab Activity 1 (Task 01) in the `BEST_MODELS` config cell below. The code defaults to `resnet50`, `densenet121`, `efficientnet_b0` as a reasonable placeholder — replace these with whichever three models actually scored best in your Task 01 comparison table.


## 0. Installation & Setup

In [ ]:
!pip install -q timm scikit-learn torchinfo thop


In [ ]:
import os, time, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision import transforms
import timm

import cv2

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


### Configuration
Edit `BEST_MODELS` to match the top-3 models from your Lab Activity 1 results.
`FILTERS` lists the five required filters plus the unfiltered baseline (`none`).

In [ ]:
# --- Fill this in with YOUR Lab 1 results ---
BEST_MODELS = {
    "Best Model 1": "resnet50",
    "Best Model 2": "densenet121",
    "Best Model 3": "efficientnet_b0",
}

FILTERS = ["none", "average", "gaussian", "median", "sharpen", "sobel"]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10          # increase for a real run; kept small so the whole pipeline finishes in one sitting
LR = 1e-4
NUM_CLASSES = 7          # HAM10000 has 7 diagnostic classes
DATA_DIR = "/content/HAM10000"     # change if your dataset lives elsewhere
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


## 1. Dataset Preparation
Loads the HAM10000 metadata CSV + image folders, reports the class distribution, and builds
train/val/test splits that are **reused identically** across every experiment (same split,
same preprocessing) so the filter comparison is fair.


In [ ]:
# HAM10000 ships as: HAM10000_metadata.csv, HAM10000_images_part_1/, HAM10000_images_part_2/
metadata_path = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
df_meta = pd.read_csv(metadata_path)

# Map each image_id to its file path (images are split across two folders)
img_dirs = [os.path.join(DATA_DIR, "HAM10000_images_part_1"),
            os.path.join(DATA_DIR, "HAM10000_images_part_2")]

def find_image_path(image_id):
    for d in img_dirs:
        p = os.path.join(d, image_id + ".jpg")
        if os.path.exists(p):
            return p
    return None

df_meta["path"] = df_meta["image_id"].apply(find_image_path)
df_meta = df_meta.dropna(subset=["path"]).reset_index(drop=True)

label_map = {name: i for i, name in enumerate(CLASS_NAMES)}
df_meta["label"] = df_meta["dx"].map(label_map)

print("Total images:", len(df_meta))
df_meta["dx"].value_counts()


In [ ]:
# Class distribution plot
plt.figure(figsize=(8,4))
sns.countplot(data=df_meta, x="dx", order=df_meta["dx"].value_counts().index)
plt.title("HAM10000 Class Distribution")
plt.xlabel("Diagnosis class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()


In [ ]:
# Fixed, reproducible train/val/test split (used for every model x filter combination)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_meta, test_size=0.30, stratify=df_meta["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))


## 2. Image Filtering
Each filter is implemented with OpenCV and applied to the image **before** it is resized/normalized and handed to the model.

In [ ]:
def apply_filter(img, filter_name):
    """img: HxWx3 uint8 RGB numpy array. Returns filtered image, same shape/dtype."""
    if filter_name == "none":
        return img
    if filter_name == "average":
        return cv2.blur(img, (5, 5))
    if filter_name == "gaussian":
        return cv2.GaussianBlur(img, (5, 5), sigmaX=1.0)
    if filter_name == "median":
        return cv2.medianBlur(img, 5)
    if filter_name == "sharpen":
        kernel = np.array([[0, -1, 0],
                            [-1, 5, -1],
                            [0, -1, 0]])
        return cv2.filter2D(img, -1, kernel)
    if filter_name == "sobel":
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        mag = np.sqrt(sx**2 + sy**2)
        mag = np.uint8(255 * mag / (mag.max() + 1e-8))
        return cv2.cvtColor(mag, cv2.COLOR_GRAY2RGB)
    raise ValueError(f"Unknown filter: {filter_name}")


In [ ]:
# Visual example: original vs each filter, on one sample image
sample_path = train_df.iloc[0]["path"]
sample_img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(FILTERS), figsize=(4*len(FILTERS), 4))
for ax, f in zip(axes, FILTERS):
    ax.imshow(apply_filter(sample_img, f))
    ax.set_title(f)
    ax.axis("off")
plt.tight_layout()
plt.savefig("filter_examples.png", dpi=150)
plt.show()


## 3. Dataset class + Model Loading

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

class HAM10000Dataset(Dataset):
    def __init__(self, df, filter_name="none", augment=False):
        self.df = df.reset_index(drop=True)
        self.filter_name = filter_name
        self.augment = augment
        aug_list = []
        if augment:
            aug_list += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(15)]
        aug_list += [transforms.ToTensor(), normalize]
        self.post = transforms.Compose(aug_list)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = apply_filter(img, self.filter_name)
        img = self.post(img)
        return img, int(row["label"])

def build_dataloaders(filter_name):
    train_ds = HAM10000Dataset(train_df, filter_name=filter_name, augment=True)
    val_ds   = HAM10000Dataset(val_df,   filter_name=filter_name, augment=False)
    test_ds  = HAM10000Dataset(test_df,  filter_name=filter_name, augment=False)
    return (DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
            DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
            DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2))


In [ ]:
def load_model(model_name, num_classes=NUM_CLASSES):
    """Loads a pretrained backbone via timm and swaps the classifier head."""
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    return model.to(device)


## 4. Training loop
Same optimizer, loss, epochs, and schedule are reused for every model x filter combination so results are comparable.

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=LR, tag=""):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                loss = criterion(out, labels)
                v_loss += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total += labels.size(0)
        val_loss = v_loss / v_total
        val_acc = v_correct / v_total

        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        print(f"[{tag}] Epoch {epoch+1}/{num_epochs} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
              f"| val_loss {val_loss:.4f} acc {val_acc:.4f}")

    return model, history


## 5. Evaluation

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            probs = torch.softmax(out, dim=1).cpu().numpy()
            preds = probs.argmax(1)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = np.nan

    per_class = classification_report(all_labels, all_preds, target_names=CLASS_NAMES,
                                       output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    metrics = {
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1,
        "Macro-F1": macro_f1, "Balanced-Accuracy": bal_acc, "AUC": auc
    }
    return metrics, cm, per_class


In [ ]:
def plot_confusion_matrix(cm, title, fname):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.show()

def plot_curves(history, title, fname_prefix):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_acc"], label="train")
    axes[0].plot(epochs, history["val_acc"], label="val")
    axes[0].set_title(f"{title} — Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(epochs, history["train_loss"], label="train")
    axes[1].plot(epochs, history["val_loss"], label="val")
    axes[1].set_title(f"{title} — Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"{fname_prefix}_curves.png", dpi=150)
    plt.show()


## 6. Baseline + Filtered Experiments (main loop)
Runs every (model x filter) combination with identical settings and collects all results into one table.
This is the cell that actually reproduces the required comparison table.


In [ ]:
results = []
histories = {}

for model_label, model_name in BEST_MODELS.items():
    for filt in FILTERS:
        tag = f"{model_label} | {filt}"
        print("="*80)
        print("Running:", tag)

        train_loader, val_loader, test_loader = build_dataloaders(filt)
        model = load_model(model_name)

        model, history = train_model(model, train_loader, val_loader, tag=tag)
        histories[tag] = history

        metrics, cm, per_class = evaluate_model(model, test_loader)

        row = {
            "Model": model_label,
            "Backbone": model_name,
            "Filter": "No Filter" if filt == "none" else filt.capitalize(),
        }
        row.update(metrics)
        results.append(row)

        safe_tag = tag.replace(" ", "_").replace("|", "-")
        plot_confusion_matrix(cm, tag, f"cm_{safe_tag}.png")
        plot_curves(history, tag, f"hist_{safe_tag}")

        # free GPU memory between runs
        del model
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv("lab02_results.csv", index=False)
results_df


## 7. Comparative Analysis

In [ ]:
pivot = results_df.pivot_table(index=["Model","Backbone"], columns="Filter", values="Accuracy")
pivot


In [ ]:
# Change relative to "No Filter" baseline, for every metric of interest
metric_cols = ["Accuracy", "Macro-F1", "Balanced-Accuracy", "AUC"]

delta_rows = []
for model_label in BEST_MODELS.keys():
    base = results_df[(results_df["Model"] == model_label) & (results_df["Filter"] == "No Filter")].iloc[0]
    for _, r in results_df[results_df["Model"] == model_label].iterrows():
        if r["Filter"] == "No Filter":
            continue
        d = {"Model": model_label, "Filter": r["Filter"]}
        for m in metric_cols:
            d[f"Delta_{m}"] = r[m] - base[m]
        delta_rows.append(d)

delta_df = pd.DataFrame(delta_rows)
delta_df


In [ ]:
# Visualize accuracy across filters for each model
plt.figure(figsize=(9,5))
for model_label in BEST_MODELS.keys():
    sub = results_df[results_df["Model"] == model_label]
    plt.plot(sub["Filter"], sub["Accuracy"], marker="o", label=model_label)
plt.title("Accuracy vs Filter, per model")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_vs_filter.png", dpi=150)
plt.show()


In [ ]:
# Which filter causes the biggest change (any direction) vs baseline, per model
delta_df["Abs_Delta_Accuracy"] = delta_df["Delta_Accuracy"].abs()
biggest_change = delta_df.sort_values("Abs_Delta_Accuracy", ascending=False).groupby("Model").head(1)
biggest_change


## 8. Answers to Lab Questions

Fill these in once `results_df` / `delta_df` above are populated with your actual run. Each answer should point back to a specific number in your results table.

1. **Which three pretrained models performed best in Lab Activity 1?**
   `_TODO: state the three models and their Task-01 accuracy/F1_`

2. **How does filtering affect each of the three models?**
   `_TODO: summarize per-model trend from the "Accuracy vs Filter" plot_`

3. **Which filter produces the greatest change compared with the unfiltered baseline?**
   `_TODO: read off "biggest_change" table above_`

4. **Does the effect of a filter remain consistent across all three models?**
   `_TODO: compare sign/magnitude of Delta_Accuracy across models for the same filter_`

5. **Does filtering improve or decrease macro-F1 and balanced accuracy?**
   `_TODO: check Delta_Macro-F1 / Delta_Balanced-Accuracy columns_`

6. **Which lesion classes are most affected by filtering?**
   `_TODO: compare per-class precision/recall (per_class dict) between "none" and each filter, per model_`

7. **Why might smoothing remove useful lesion texture or morphological information?**
   Average/Gaussian/Median filters attenuate high-frequency detail — the same frequencies that carry lesion border irregularity, pigment network texture, and fine structures (dots/globules) that distinguish malignant from benign lesions. Smoothing them out can blur the very cues the classifier relies on.

8. **Why might sharpening or edge detection help or hurt classification?**
   Sharpening/Sobel emphasize edges and boundaries, which can help highlight lesion borders (useful for asymmetry/border-irregularity cues) but can also amplify noise, hair artifacts, or irrelevant skin texture, and (for Sobel) discards color/intensity information entirely, which pretrained CNNs otherwise rely on heavily.

9. **What is the difference between convolution and correlation?**
   Correlation slides a kernel over the image and computes a weighted sum directly; convolution additionally flips the kernel (180°) before sliding it. For symmetric kernels (e.g. a Gaussian or a box filter) the two are identical; for asymmetric kernels (e.g. many edge/derivative kernels) they differ. In practice, most deep-learning frameworks (including PyTorch's `nn.Conv2d`) implement cross-correlation rather than true convolution, since the kernel weights are learned anyway and the flip has no effect on what the network can represent.

10. **Relationship between classical image processing and deep-learning-based feature extraction:**
    `_TODO: tie back to your specific results — e.g., if filtering had little/negative effect, that suggests the CNN backbone already learns comparable (or better) feature-extraction filters end-to-end in its early convolutional layers, making hand-crafted preprocessing largely redundant or even harmful; if a specific filter helped, discuss why that inductive bias wasn't already captured by the pretrained features._
